In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
SCRIPT_DIR = Path.cwd()
DATA_PATH = SCRIPT_DIR / "USSeatBelts.csv"
if not DATA_PATH.exists():
    DATA_PATH = SCRIPT_DIR.parent / "USSeatBelts.csv"
def load_seatbelts():
    if not DATA_PATH.exists():
        raise FileNotFoundError(
            f"Could not find {DATA_PATH.resolve()}. Put USSeatBelts.csv in this folder "
            "or change DATA_PATH at the top of this script."
        )
    return pd.read_csv(DATA_PATH)
def show_data(df):
    print("\nSHAPE:", df.shape)
    print("\nCOLUMNS:", list(df.columns))
    print("\nINFO:")
    df.info()
    print("\nFIRST SIX ROWS:\n", df.head(6))
    print("\nLAST SIX ROWS:\n", df.tail(6))
    print("\nSUMMARY:\n", df.describe(include="all").T)
df = load_seatbelts()
show_data(df)


SHAPE: (765, 13)

COLUMNS: ['rownames', 'state', 'year', 'miles', 'fatalities', 'seatbelt', 'speed65', 'speed70', 'drinkage', 'alcohol', 'income', 'age', 'enforce']

INFO:
<class 'pandas.DataFrame'>
RangeIndex: 765 entries, 0 to 764
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   rownames    765 non-null    int64  
 1   state       765 non-null    str    
 2   year        765 non-null    int64  
 3   miles       765 non-null    int64  
 4   fatalities  765 non-null    float64
 5   seatbelt    556 non-null    float64
 6   speed65     765 non-null    str    
 7   speed70     765 non-null    str    
 8   drinkage    765 non-null    str    
 9   alcohol     765 non-null    str    
 10  income      765 non-null    int64  
 11  age         765 non-null    float64
 12  enforce     765 non-null    str    
dtypes: float64(3), int64(4), str(6)
memory usage: 77.8 KB

FIRST SIX ROWS:
    rownames state  year  miles  fatalitie

In [2]:
print("\nMISSING VALUES BY COLUMN\n", df.isna().sum())
print("\nTOTAL MISSING VALUES:", df.isna().sum().sum())
print("\nROWS WITH MISSING VALUES\n", df[df.isna().any(axis=1)].head(10))
clean_rows = df.dropna()
print("\nAFTER DROPPING ROWS WITH MISSING VALUES:", clean_rows.shape)
# Separate examples, each starts from the original data.
mean_imputed = df.copy()
if "income" in mean_imputed:
    mean_imputed["income"] = mean_imputed["income"].fillna(mean_imputed["income"].mean())
median_imputed = df.copy()
if "age" in median_imputed:
    median_imputed["age"] = median_imputed["age"].fillna(median_imputed["age"].median())
mode_imputed = df.copy()
if "seatbelt" in mode_imputed and not mode_imputed["seatbelt"].mode().empty:
    mode_imputed["seatbelt"] = mode_imputed["seatbelt"].fillna(mode_imputed["seatbelt"].mode()[0])
print("\nMEAN-IMPUTED INCOME MISSING:", mean_imputed.get("income", pd.Series(dtype=float)).isna().sum())
print("MEDIAN-IMPUTED AGE MISSING:", median_imputed.get("age", pd.Series(dtype=float)).isna().sum())
print("MODE-IMPUTED SEATBELT MISSING:", mode_imputed.get("seatbelt", pd.Series(dtype=float)).isna().sum())
ffill = df.ffill()
bfill = df.bfill()
print("\nFORWARD-FILL SAMPLE\n", ffill.head())
print("\nBACKWARD-FILL SAMPLE\n", bfill.head())
filled = mode_imputed.copy()
filled = filled.fillna({c: "Unknown" for c in filled.select_dtypes(exclude=np.number).columns})
print("\nDUPLICATE ROWS:", filled.duplicated().sum())
filled = filled.drop_duplicates()
print("SHAPE AFTER DEDUPLICATION:", filled.shape)


MISSING VALUES BY COLUMN
 rownames        0
state           0
year            0
miles           0
fatalities      0
seatbelt      209
speed65         0
speed70         0
drinkage        0
alcohol         0
income          0
age             0
enforce         0
dtype: int64

TOTAL MISSING VALUES: 209

ROWS WITH MISSING VALUES
     rownames state  year  miles  fatalities  seatbelt speed65 speed70  \
0          1    AK  1983   3358    0.044669       NaN      no      no   
1          2    AK  1984   3589    0.037336       NaN      no      no   
2          3    AK  1985   3840    0.033073       NaN      no      no   
3          4    AK  1986   4008    0.025200       NaN      no      no   
4          5    AK  1987   3900    0.019487       NaN      no      no   
5          6    AK  1988   3841    0.025254       NaN      no      no   
6          7    AK  1989   3887    0.021610       NaN      no      no   
15        16    AL  1983  31032    0.029969       NaN      no      no   
30        31   

In [3]:
if "fatalities" in filled.columns:
    filled = filled.rename(columns={"fatalities":"fatality_rate"})
if {"miles","fatality_rate"}.issubset(filled.columns):
    filled["miles_per_fatality"] = filled["miles"] / filled["fatality_rate"].replace(0, np.nan)
print("\nDERIVED DATA SAMPLE\n", filled.head())
if "income" in df:
    x = pd.to_numeric(df["income"], errors="coerce").dropna()
    print("\nINCOME MIN/MAX/RANGE:", x.min(), x.max(), (x.min(), x.max()))
    print("INCOME VARIANCE:", x.var(), "SD:", x.std(), "IQR:", x.quantile(.75)-x.quantile(.25))


DERIVED DATA SAMPLE
    rownames state  year  miles  fatality_rate  seatbelt speed65 speed70  \
0         1    AK  1983   3358       0.044669      0.58      no      no   
1         2    AK  1984   3589       0.037336      0.58      no      no   
2         3    AK  1985   3840       0.033073      0.58      no      no   
3         4    AK  1986   4008       0.025200      0.58      no      no   
4         5    AK  1987   3900       0.019487      0.58      no      no   

  drinkage alcohol  income        age enforce  miles_per_fatality  
0      yes      no   17973  28.234966      no        75174.427499  
1      yes      no   18093  28.343542      no        96126.277648  
2      yes      no   18925  28.372816      no       116107.092717  
3      yes      no   18466  28.396652      no       159050.134648  
4      yes      no   18021  28.453251      no       200131.575573  

INCOME MIN/MAX/RANGE: 8372 35863 (np.int64(8372), np.int64(35863))
INCOME VARIANCE: 23150140.554279167 SD: 4811.459295